# Cat vs Dog Image Classification

This project uses a **Convolutional Neural Network (CNN)** built with PyTorch to classify images into two categories: cats and dogs.

The notebook covers image preprocessing, dataset loading, CNN model creation, model training, validation, saving/loading the model, and making a prediction on a new image.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 1. Import Libraries

Import the libraries required for image processing, building and training the CNN, visualization, and model evaluation.

In [ ]:

train_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

test_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

## 2. Image Preprocessing

Resize images to **224 × 224 pixels**, apply data augmentation to training images, convert images to tensors, and normalize pixel values. The test transformation does not apply random augmentation.

In [ ]:
train_dataset = datasets.ImageFolder(

    root="data1/training_set",

    transform=train_transform

)

val_dataset = datasets.ImageFolder(
    root = "data1/validation_set",
    transform=train_transform
)

test_dataset = datasets.ImageFolder(

    root="data1/test_set",

    transform=test_transform

)

## 3. Load the Dataset

Load the cat-and-dog images using `ImageFolder`. The dataset is divided into training, validation, and test folders. Each split should contain separate `cats` and `dogs` subfolders.

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(

    train_dataset,

    batch_size=32,

    shuffle=True,

    num_workers=4,

    pin_memory=True
)

val_loader = DataLoader(
    
    val_dataset,

    batch_size=32,

    shuffle=False
)

test_loader = DataLoader(

    test_dataset,

    batch_size=32,

    shuffle=False,

    num_workers=4,

    pin_memory=True

)

## 4. Create DataLoaders

DataLoaders load images in batches. The training loader shuffles the images to support learning, while validation and test loaders keep a fixed order.

In [ ]:
images, labels = next(iter(train_loader))

print(images.shape)

print(labels.shape)

## 5. Check a Batch of Images

Load one batch from the training DataLoader and inspect the image and label tensor dimensions.

In [ ]:
import torch.nn as nn

model = nn.Sequential(

    nn.Conv2d(3,16,3,padding=1),

    nn.ReLU(),

    nn.MaxPool2d(2),

    nn.Conv2d(16,32,3,padding=1),

    nn.ReLU(),

    nn.MaxPool2d(2),

    nn.Flatten(),

    nn.Linear(32*56*56,128),

    nn.ReLU(),

    nn.Linear(128,2)
)

## 6. Build the CNN Model

Create a simple CNN using convolutional layers, ReLU activation, max-pooling, flattening, and fully connected layers. The final layer returns two output scores: one for each class.

In [ ]:
import matplotlib.pyplot as plt

image = images[0]

image = image.permute(1,2,0)

plt.imshow(image)

plt.show()

## 7. Visualize a Sample Image

Display one image from the training batch to inspect the data being supplied to the model.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

## 8. Define the Loss Function and Optimizer

Use **CrossEntropyLoss** to measure classification error and the **Adam optimizer** to update the model weights during training.

In [ ]:
for epoch in range(10):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}: {running_loss:.4f}")

## 9. Train the CNN Model

Train the model for 10 epochs. In each epoch, the model processes the training batches, calculates the loss, performs backpropagation, and updates its parameters.

In [ ]:
model.eval()

correct = 0

total = 0

with torch.no_grad():

    for images, labels in val_loader:

        outputs = model(images)

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"validation Accuracy: {accuracy:.2f}%")

## 10. Validate the Model

Evaluate the trained model on the validation dataset without updating its weights. Calculate validation accuracy by comparing predicted labels with actual labels.

In [ ]:
# saving the model
import torch

torch.save(
    model.state_dict(),
    "cat_dog_model.pth"
)

## 11. Save the Trained Model

Save the trained model parameters to a `.pth` file so the model can be reused without training from the beginning.

In [ ]:
# loading the model

model.load_state_dict(
    torch.load("cat_dog_model.pth")
)

model.eval()

## 12. Load the Saved Model

Load the saved model parameters back into the CNN and switch the model to evaluation mode.

In [ ]:
from PIL import Image

image = Image.open("data1/test_set/cats/cat.4546.jpg")

image = train_transform(image)

image = image.unsqueeze(0)

model.eval()

with torch.no_grad():

    output = model(image)

    prediction = output.argmax(1)

print(prediction)

## 14. Final Results and Model Evaluation

Evaluate the saved model on the **test dataset**. This section displays the final test accuracy, confusion matrix, and classification report. Run this cell after the model has been trained and loaded. The actual results will appear below when you execute the notebook.

In [ ]:
model.eval()
all_labels = []
all_predictions = []
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

test_accuracy = 100 * correct / total
print(f"Test Accuracy: {test_accuracy:.2f}%")

print("\nClassification Report:")
print(classification_report(all_labels, all_predictions, target_names=test_dataset.classes))

cm = confusion_matrix(all_labels, all_predictions)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=test_dataset.classes,
            yticklabels=test_dataset.classes)
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Cat vs Dog - Confusion Matrix")
plt.tight_layout()
plt.show()

## 15. Conclusion

In this project, a CNN was developed using PyTorch to classify cat and dog images. The model was trained using the training dataset, checked using the validation dataset, and evaluated using the test dataset. The final test accuracy and classification metrics should be recorded here after running the notebook.

**Note:** Do not enter an estimated accuracy. Use the values generated by your own run.